<a href="https://colab.research.google.com/github/Sangeetha3315/Agentic-AI-and-computer-vision-workshop-projects/blob/main/vision_detective.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q transformers torch torchvision accelerate pillow matplotlib numpy

import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

DEVICE = 0 if torch.cuda.is_available() else -1
device_str = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device_str)

# --- Lighter settings for CPU-only environments ---
TORCH_DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
LOW_MEMORY = not torch.cuda.is_available()

if LOW_MEMORY:
    print("No GPU detected — using lightweight/CPU-friendly configuration.")
    print("Tip: prefer smaller model checkpoints (e.g. 'distilbert', 'MobileViT', "
          "'tiny'/'small' variants) to keep inference fast on CPU.")

In [ ]:
import torch
from transformers import (
    BlipProcessor, BlipForConditionalGeneration, BlipForQuestionAnswering,
    pipeline,
)

device = "cuda" if torch.cuda.is_available() else "cpu"   # for BLIP
PIPE_DEVICE = 0 if torch.cuda.is_available() else -1        # for pipeline() calls

print("Loading captioner (BLIP)...")
caption_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
caption_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)

print("Loading visual question-answering model (BLIP-VQA)...")
vqa_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")
vqa_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base").to(device)

print("Loading object detector (DETR)...")
detector = pipeline("object-detection", model="facebook/detr-resnet-50", device=PIPE_DEVICE)

print("Loading segmentation model (SegFormer)...")
segmenter = pipeline("image-segmentation", model="nvidia/segformer-b0-finetuned-ade-512-512", device=PIPE_DEVICE)

print("\nAll four models loaded.")

In [ ]:
import requests
from PIL import Image
from io import BytesIO

url = "https://images.unsplash.com/photo-1548199973-03cce0bbc87b?w=800"  # a street scene
response = requests.get(url)
image = Image.open(BytesIO(response.content)).convert("RGB")
image

In [ ]:
inputs = caption_processor(image, return_tensors="pt").to(device)
out = caption_model.generate(**inputs, max_new_tokens=50)
caption = caption_processor.decode(out[0], skip_special_tokens=True)

print("Caption:", caption)

In [ ]:
def ask(image, question):
    inputs = vqa_processor(image, question, return_tensors="pt").to(device)
    out = vqa_model.generate(**inputs)
    answer = vqa_processor.decode(out[0], skip_special_tokens=True)
    print(f"Q: {question}")
    print(f"A: {answer}\n")
    return answer

# A few to start
ask(image, "How many people are in this image?")
ask(image, "Is this indoors or outdoors?")
ask(image, "What time of day does this look like?")
ask(image, "What color is the sky?")

In [ ]:
from PIL import ImageDraw

def draw_detections(pil_image, detections, threshold=0.7):
    img = pil_image.copy()
    draw = ImageDraw.Draw(img)
    colors = ["#B8503F", "#6E7F5C", "#D9A441", "#8F3D2F", "#51603F", "#3A6EA5"]
    for i, det in enumerate(detections):
        if det["score"] < threshold:
            continue
        box = det["box"]
        x0, y0, x1, y1 = box["xmin"], box["ymin"], box["xmax"], box["ymax"]
        color = colors[i % len(colors)]
        draw.rectangle([x0, y0, x1, y1], outline=color, width=4)
        label = f"{det['label']} {det['score']:.2f}"
        tb = draw.textbbox((x0, y0), label)
        draw.rectangle([x0, y0 - (tb[3]-tb[1]) - 6, tb[2]-tb[0]+x0+8, y0], fill=color)
        draw.text((x0 + 4, y0 - (tb[3]-tb[1]) - 4), label, fill="white")
    return img

detections = detector(image)
draw_detections(image, detections, threshold=0.7)

In [ ]:
import numpy as np

def draw_segmentation(pil_image, seg_results, alpha=0.55):
    base = pil_image.convert("RGBA")
    overlay = Image.new("RGBA", base.size, (0, 0, 0, 0))
    rng = np.random.default_rng(7)
    palette = {}

    for seg in seg_results:
        label = seg["label"]
        if label not in palette:
            palette[label] = tuple(rng.integers(60, 255, size=3).tolist()) + (int(255 * alpha),)
        mask = seg["mask"].convert("L")
        color_layer = Image.new("RGBA", base.size, palette[label])
        overlay.paste(color_layer, (0, 0), mask)

    combined = Image.alpha_composite(base, overlay).convert("RGB")
    print("Regions found:", ", ".join(palette.keys()))
    return combined

seg_results = segmenter(image)
draw_segmentation(image, seg_results)

In [ ]:
import matplotlib.pyplot as plt

FUN_QUESTIONS = [
    "How many people are in this image?",
    "Is this indoors or outdoors?",
    "What is the main subject of this image?",
    "What mood does this image convey?",
]

def caption_image(pil_image):
    inputs = caption_processor(pil_image, return_tensors="pt").to(device)
    out = caption_model.generate(**inputs, max_new_tokens=50)
    return caption_processor.decode(out[0], skip_special_tokens=True)

def ask(pil_image, question):
    inputs = vqa_processor(pil_image, question, return_tensors="pt").to(device)
    out = vqa_model.generate(**inputs)
    return vqa_processor.decode(out[0], skip_special_tokens=True)

def vision_detective_report(pil_image, extra_questions=None):
    cap = caption_image(pil_image)
    dets = detector(pil_image)
    segs = segmenter(pil_image)

    boxed = draw_detections(pil_image, dets, threshold=0.7)
    painted = draw_segmentation(pil_image, segs)

    questions = FUN_QUESTIONS + (extra_questions or [])
    answers = [(q, ask(pil_image, q)) for q in questions]

    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    axes[0].imshow(boxed); axes[0].set_title("Detected objects"); axes[0].axis("off")
    axes[1].imshow(painted); axes[1].set_title("Segmented regions"); axes[1].axis("off")
    fig.suptitle(f'"{cap}"', fontsize=14, style="italic")
    plt.tight_layout()
    plt.show()

    print("🔎 Detective's Q&A:")
    for q, a in answers:
        print(f"   {q}  →  {a}")

vision_detective_report(image)

In [ ]:
from google.colab import files

uploaded = files.upload()
fname = list(uploaded.keys())[0]
my_image = Image.open(fname).convert("RGB")

vision_detective_report(my_image, extra_questions=["What is unusual about this image?"])

In [ ]:
from IPython.display import display, Javascript
from google.colab.output import eval_js
from base64 import b64decode

def take_photo(filename="webcam.jpg", quality=0.9):
    js = Javascript('''
        async function takePhoto(quality) {
            const div = document.createElement('div');
            const capture = document.createElement('button');
            capture.textContent = 'Capture';
            div.appendChild(capture);

            const video = document.createElement('video');
            video.style.display = 'block';
            const stream = await navigator.mediaDevices.getUserMedia({video: true});

            document.body.appendChild(div);
            div.appendChild(video);
            video.srcObject = stream;
            await video.play();

            google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

            await new Promise((resolve) => capture.onclick = resolve);

            const canvas = document.createElement('canvas');
            canvas.width = video.videoWidth;
            canvas.height = video.videoHeight;
            canvas.getContext('2d').drawImage(video, 0, 0);
            stream.getVideoTracks()[0].stop();
            div.remove();
            return canvas.toDataURL('image/jpeg', quality);
        }
        ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    with open(filename, 'wb') as f:
        f.write(binary)
    return filename

photo_path = take_photo()
webcam_image = Image.open(photo_path).convert("RGB")
vision_detective_report(webcam_image, extra_questions=["What is the person doing?"])